In [1]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [4]:
import ollama
response = ollama.chat(model="llama3.1:8b",
                       messages=[{"role": "user", "content": "Say hello in one sentence."}])
print(response["message"]["content"])

Hello, how can I assist you today?


In [1]:
import chromadb

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("lufthansa")     # reopen the existing store

print("Loaded collection with", collection.count(), "docs")

Loaded collection with 200 docs


In [4]:
question = "What are the biggest risks for Lufthansa right now?"

# retrieve the 5 most relevant docs (semantic search)
results   = collection.query(query_texts=[question], n_results=5)
retrieved = results["documents"][0]
metas     = results["metadatas"][0]

# build ONE context string from the retrieved docs (tag each with its source)
context = "\n\n".join(f"[{m['source']}] {doc}" for doc, m in zip(retrieved, metas))

print(context)

[news] Deutsche Lufthansa : Lufthansa CEO sees. | MarketScreener. Lufthansa expects the level of business travel in the medium term to be around 90% or more of what it was before COVID-19.

[news] Current information | Lufthansa. Get the latest flight information including updates on routes, cancellations due to weather conditions or strikes, rebooking options, and more.

[news] Lufthansa narrows losses in first quarter as demand offsets rising fuel . Lufthansa Group reported an improved financial performance in the first quarter of 2026, with revenue rising 8% to €8.7 billion while operating losses narrowed compared to the same period last year. The group posted an adjusted operating loss (EBIT) of €612 million, compared with a loss of €722 million a year earlier. In practical terms, the company is still in the red for the quarter .

[news] Lufthansa sticks to 2026 outlook despite $2 billion jet-fuel hit . Lufthansa kept its 2026 profit outlook on Wednesday, saying hedging, higher far

testing ollama with correct system_prompt and user_prompt

In [5]:
import ollama

system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided to answer — do not invent facts.
Be concise, specific, and ground every claim in the evidence."""

user_prompt = f"""Evidence:
{context}

Question: {question}

Answer as a strategic advisor, citing the evidence."""

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
)

print(response["message"]["content"])

Based on the provided evidence, I identify two significant risks for Lufthansa:

1. **Recovery of demand and resilience to future disruptions**: Although Lufthansa expects business travel to reach 90% or more of pre-COVID-19 levels in the medium term (Source 1), this assumption is based on a "medium-term" perspective, which implies some uncertainty about short-term fluctuations. Moreover, the company has faced significant challenges due to weather conditions and strikes, as mentioned in Source 2. These events can still impact demand and revenue.
2. **Fuel price volatility**: Lufthansa faces a $2 billion jet-fuel hit to costs (Source 3). While hedging strategies will help mitigate this risk, the company's profit outlook remains uncertain due to the potential for future fuel price fluctuations.

These two risks are interconnected, as increased demand and revenue can be offset by higher fuel prices. A more stable and predictable revenue stream would help Lufthansa better navigate these ch

In [6]:
import json, ollama

system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "supporting_evidence": list of 2-3 short evidence points taken from the context
- "expected_impact": the expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""

user_prompt = f"""Evidence:
{context}

Question: {question}"""

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ],
    format="json"          # ← forces Ollama to output valid JSON
)

rec = json.loads(response["message"]["content"])   # parse JSON string → Python dict
print(json.dumps(rec, indent=2))                   # pretty-print it

{
  "recommendation": "Maintain cost-cutting efforts and explore hedging strategies",
  "supporting_evidence": [
    "Lufthansa Group reported an adjusted operating loss (EBIT) of \u20ac612 million in Q1 2026, compared to a loss of \u20ac722 million a year earlier.",
    "The group posted a 1.7 billion euro ($2 billion) jet-fuel hit to costs in Q1 2026",
    "Lufthansa kept its 2026 profit outlook on Wednesday, saying hedging, higher fares and cost cuts would help offset the jet-fuel hit"
  ],
  "expected_impact": "Improved financial performance and reduced losses",
  "risk_level": "Medium",
  "priority": "High"
}


reusable ceo_agent(question) function for my dashboard

In [9]:
def ceo_agent(question, k=5):
    # 1. RETRIEVE evidence
    results   = collection.query(query_texts=[question], n_results=k)
    retrieved = results["documents"][0]
    metas     = results["metadatas"][0]
    context   = "\n\n".join(f"[{m['source']}] {doc}" for doc, m in zip(retrieved, metas))

    # 2. PROMPT
    system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "justification": 1-2 sentences explaining WHY this recommendation follows from the evidence
- "supporting_evidence": list of 2-3 short evidence points from the context
- "expected_impact": expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""
    user_prompt = f"Evidence:\n{context}\n\nQuestion: {question}"

    # 3. GENERATE (structured JSON)
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )

    # 4. PARSE + attach question and evidence URLs
    rec = json.loads(response["message"]["content"])
    rec["question"] = question
    rec["sources"]  = [m["url"] for m in metas]
    return rec

testing reusable code

In [8]:
import json
result = ceo_agent("What are the major opportunities for Lufthansa?")
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "recommendation": "Invest in expanding route network to capitalize on expected business travel recovery",
  "supporting_evidence": [
    "Lufthansa expects the level of business travel in the medium term to be around 90% or more of what it was before COVID-19.",
    "Lufthansa has a large route network with destinations from A (Abidjan) to Z (Zurich).",
    "The company is the largest airline in Europe, both in terms of overall passengers carried and fleet size."
  ],
  "expected_impact": "Significant revenue growth as business travel recovers",
  "risk_level": "Medium",
  "priority": "High",
  "question": "What are the major opportunities for Lufthansa?",
  "sources": [
    "https://www.marketscreener.com/quote/stock/LUFTHANSA-436827/news/Deutsche-Lufthansa-Lufthansa-CEO-sees-business-travel-recovering-faster-than-thought-36769814/",
    "https://business.lufthansagroup.com/us/en/program/experts/airlines-at-a-glance/lufthansa",
    "https://www.reddit.com/r/AviationPorn/comments/f

In [10]:
questions = [
    "What are the major opportunities for Lufthansa?",
    "What are the biggest risks for Lufthansa?",
    "What are competitors doing?",
    "Which technologies or trends should Lufthansa management monitor?",
    "What strategic actions should Lufthansa prioritize?",
]

recommendations = []
for q in questions:
    print("Generating:", q)
    recommendations.append(ceo_agent(q))

json.dump(recommendations,
          open("recommendations.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\n✅ Saved", len(recommendations), "recommendations")

Generating: What are the major opportunities for Lufthansa?
Generating: What are the biggest risks for Lufthansa?
Generating: What are competitors doing?
Generating: Which technologies or trends should Lufthansa management monitor?
Generating: What strategic actions should Lufthansa prioritize?

✅ Saved 5 recommendations


CEO Briefing (Section 7)

In [11]:
def ceo_briefing(recommendations):
    # summarize the recommendations as input
    rec_summary = "\n".join(
        f"- {r['recommendation']} (priority {r['priority']}, risk {r['risk_level']})"
        for r in recommendations
    )

    system_prompt = """You are chief of staff to the CEO of Lufthansa.
Write a concise executive briefing as a JSON object with EXACTLY these keys:
- "what_happened": key recent developments (2-3 sentences)
- "why_it_matters": why these developments matter to the business (2-3 sentences)
- "what_to_do_next": the top priority actions management should take (2-3 sentences)
Base it ONLY on the recommendations provided. Do not invent facts."""

    user_prompt = f"Strategic recommendations:\n{rec_summary}\n\nWrite the CEO briefing."

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )
    return json.loads(response["message"]["content"])

In [12]:
briefing = ceo_briefing(recommendations)
json.dump(briefing, open("ceo_briefing.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(json.dumps(briefing, indent=2, ensure_ascii=False))

{
  "what_happened": {
    "long-haul flights": "Demand for long-haul travel continues to grow, prompting a high-priority recommendation to increase our capacity",
    "digital transformation": "Investments are needed in digital initiatives to enhance customer experience and operational efficiency",
    "competitive landscape": "Competitors such as United Airlines, Qatar Airways, and Turkish Airlines are actively updating fleets and integrating ITA systems, necessitating close monitoring"
  },
  "why_it_matters": {
    "demand growth": "Meeting growing demand for long-haul flights is crucial to maintaining market share",
    "customer experience": "Digital transformation will drive customer satisfaction and loyalty",
    "competitive parity": "Monitoring the competitive landscape ensures Lufthansa stays abreast of industry developments"
  },
  "what_to_do_next": {
    "increase long-haul capacity": "Allocate resources to increase long-haul flights in response to growing demand",
    "i